# Clasificación por Red Neuronal

#### Autor: Erick Salas Chaverri

#### Objetivo:
Mostrar la ejercución de una red neuronal para clasificar puntos según sus ubicación en el plano.

Se va a crear dataset con información de la ubicación de puntos en un plano

Se van a realizar los siguiente pasos:

- Creación
- Caracterización
- Transformación
- Minería de datos: Aplicación de la red neuronal para clasificar
- Predición


## **Pasos**

## Importación de las bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns                     # Para graficar
import plotly.express as px
import plotly.graph_objects as go

from pandas.plotting import scatter_matrix
from sklearn.preprocessing import MinMaxScaler

from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score

import tensorflow as tf
tf.keras.utils.set_random_seed(42)
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix


from sklearn.datasets import make_blobs   # Para generar datos dummies

# Lectura del dataset

In [ ]:
df = pd.read_csv("tree_groups.csv")
df.head()

# Caracterización del dataset

In [ ]:
# Definir el gráfico de dispersión de los puntos de las características
fig_features = px.scatter(df, x="x", y="y", color='label')


fig_features.update_traces(
    marker=dict(size=10))

fig_features.update_yaxes(
    scaleanchor="x",
    scaleratio=1,
  )

fig_features.show()

In [ ]:
# este tipo de clasificación se llama supervisado

feature_names = ['x', 'y']
# estamos filtrando la tabla para que tener sólo las columnas que necesito

# estamos obteniendo sólo las features
#              tabla   []
X = df[feature_names]


# estamos obteniendo los labels
#            tabla []
y = df['label']

In [ ]:
X.head()

In [ ]:
# dataset para entrenar . training
# dataset pruebas.        test

# X para las features
# y para los labels
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)


In [ ]:
# para tener los features bajo la misma escala de valores
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)


In [ ]:
len(X.keys())

In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical
num_classes = len(np.unique(y))
y_train_one_hot = to_categorical(y_train, num_classes=num_classes)
y_val_one_hot = to_categorical(y_val, num_classes=num_classes)
y_test_one_hot = to_categorical(y_test, num_classes=num_classes)


In [ ]:
# estamos creado la variable modelo 'modelo'
model = keras.Sequential([
  layers.Dense(64, activation='relu', input_shape=[len(X.keys())]),  # capa 1
  layers.Dense(64, activation='relu'),                               # capa 2
  layers.Dense(y.nunique(), activation='softmax')                    # capa 3
])

model.compile(optimizer = tf.optimizers.Adam(),
            loss = 'categorical_crossentropy',
            metrics=['accuracy'])


## Qué es categorical_crossentropy?

categorical_crossentropy es una función de pérdida (loss function) muy común en el entrenamiento de redes neuronales, especialmente para problemas de clasificación multiclase donde las etiquetas de clase están en formato one-hot encoding.

Aquí tienes una explicación detallada:

- ¿Qué es? Es una medida de la diferencia entre dos distribuciones de probabilidad. En el contexto de una red neuronal, mide qué tan diferente es la distribución de probabilidad predicha por el modelo de la distribución de probabilidad verdadera (real) para cada clase.

- ¿Cuándo se usa? Se utiliza cuando tienes un problema de clasificación con más de dos clases (multiclase) y tus etiquetas (y_true) están en un formato one-hot encoding. Por ejemplo, si tienes 3 clases (0, 1, 2), una etiqueta one-hot para la clase 0 sería [1, 0, 0], para la clase 1 sería [0, 1, 0], y para la clase 2 sería [0, 0, 1].

- ¿Por qué es adecuada para one-hot encoding? La salida de la capa final de tu red neuronal, cuando usas softmax como función de activación (como en tu modelo), es una distribución de probabilidad sobre las clases. categorical_crossentropy compara directamente esta distribución predicha con la distribución one-hot encoding de la etiqueta verdadera. Si las predicciones de tu modelo son muy parecidas a las etiquetas verdaderas (one-hot), el valor de la pérdida será bajo; si son muy diferentes, la pérdida será alta.

- Diferencia con sparse_categorical_crossentropy: La principal diferencia es el formato de las etiquetas verdaderas. Si tus etiquetas son enteros (ej. 0, 1, 2) en lugar de vectores one-hot, usarías sparse_categorical_crossentropy. Esta función realiza internamente la conversión a one-hot encoding antes de calcular la entropía cruzada, ahorrándote un paso.

In [ ]:
model.summary()

In [ ]:
BATCH_SIZE=100
EPOCHS = 40

# para entrenar el modelo
historia = model.fit(X_train, y_train_one_hot, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(X_val, y_val_one_hot), verbose=1)

In [ ]:
# evaluando el modelo con los datos de entrenamiento
model.evaluate(X_train, y_train_one_hot)

In [ ]:
# evaluando el modelo con los datos de prueba test
model.evaluate(X_test, y_test_one_hot)

In [ ]:
def plot_history(history, samples=10, init_phase_samples=None):
    epochs = len(history.history['accuracy'])

    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']

    every_sample =  max(1, int(epochs / samples))
    acc = pd.DataFrame(acc).iloc[::every_sample, :]
    val_acc = pd.DataFrame(val_acc).iloc[::every_sample, :]

    fig, ax = plt.subplots(figsize=(20,5))

    ax.plot(acc, 'bo', label='Training acc')
    ax.plot(val_acc, 'b', label='Validation acc')
    ax.set_title('Precisión de entrenamineto y validación')
    ax.legend()

In [ ]:
plot_history(historia)

# Predecir con nuevos valores

In [ ]:
nuevos_datos = pd.DataFrame([
    [5, 5],
    [-5, 5],
    [-5, -5],
],
    columns=['x', 'y'])
nuevos_datos.head()

In [ ]:
# para escalado
nuevos_datos_escalados = scaler.transform(nuevos_datos)

In [ ]:
nuevos_datos_escalados

In [ ]:
# para hacer la predicción!
prediccion = model.predict(nuevos_datos_escalados)
prediccion

In [ ]:
prediccion.argmax(axis=1)

# Creación de límites de decisión

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np

In [ ]:
def plot_multiclass_decision_boundary(model, X, y):
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 101), np.linspace(y_min, y_max, 101))
    cmap = ListedColormap(['#FF0000', '#00FF00', '#0000FF'])

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()], verbose=0).argmax(axis=1)
    Z = Z.reshape(xx.shape)
    fig = plt.figure(figsize=(8, 8))
    plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.RdYlBu)
    plt.xlim(xx.min(), xx.max())
    plt.ylim(yy.min(), yy.max())

In [ ]:
plot_multiclass_decision_boundary(model, scaler.transform(X), y)